# Qwen3.5-9B Jordan Peterson Fine-Tuning
## Hybrid Architecture (Linear + Full Attention) · V4 Q&A Data · 3 Epochs · r=32 LoRA

This notebook fine-tunes **Qwen3.5-9B** on Jordan B. Peterson's four books using
the same V4 training pipeline as `Qwen3_14B_JordanPeterson_V4_FineTuning.ipynb`.

**Pipeline position:** Run `JordanPeterson_DataPrep.ipynb` first to generate
the Q&A cache (`qa_dataset/peterson_qa.jsonl`).  This notebook reads from that
cache directly — no PDF extraction or question generation happens here.

### Why Qwen3.5-9B?

Qwen3.5 is Alibaba’s latest model family (Feb 2026), featuring a hybrid
architecture that alternates **Gated DeltaNet** (linear attention) layers with
**standard full attention** layers.  The 27B variant was tested first but
**OOM'd on the RTX 4090 (24 GB)**.  The 9B variant has:

- 32 layers (24 linear attention + 8 full attention)
- Hidden dim 4096, 16 attention heads, 4 KV heads (GQA)
- 256K native context window
- Vision encoder (unused for text-only fine-tuning)

At 4-bit quantization this loads at ~8.5 GB, leaving ~16 GB for LoRA training
overhead on the RTX 4090.  We can use `batch_size=2` with `grad_accum=4`
for an effective batch size of 8 (same as all Qwen3-14B notebooks).

### Key Differences from Qwen3-14B Notebooks

| Aspect | Qwen3-14B | **Qwen3.5-9B** |
|--------|-----------|---------------|
| Loading | `FastLanguageModel` | **`FastVisionModel`** (VLM architecture) |
| Tokenizer | returned directly | **`processor.tokenizer`** |
| Architecture | Dense transformer | **Hybrid linear + full attention** |
| LoRA targets | explicit `target_modules` | **`finetune_language_layers=True`** |
| Parameters | 14B | **9B** (smaller but newer architecture) |
| Chat template | ChatML | **ChatML** (identical format) |

In [1]:
import json, re, math, time, gc
from pathlib import Path

import torch
from datasets import Dataset
from unsloth import FastVisionModel
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

# ── Load shared config ─────────────────────────────────────────────────────────────
with open("peterson_config.json") as f:
    _config = json.load(f)

QA_CACHE      = Path(_config["paths"]["qa_cache"])
SYSTEM_PROMPT = _config["system_prompt"]

# ── Model + training constants ───────────────────────────────────────────────────
BASE_MODEL    = "unsloth/Qwen3.5-9B"
OUTPUT_DIR    = Path("./outputs/qwen3_5_9b_peterson_lora")
MAX_SEQ_LEN   = 2048
LORA_RANK     = 32
LORA_ALPHA    = 32
BATCH_SIZE    = 2       # 9B at 4-bit (~8.5 GB) leaves plenty of headroom
GRAD_ACCUM    = 4       # effective batch = 2 x 4 = 8 (same as Qwen3-14B)
NUM_EPOCHS    = 3
LEARNING_RATE = 2e-4
WARMUP_STEPS  = 30
WEIGHT_DECAY  = 0.01

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}  |  GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print()
print("Configuration:")
print(f"  Base model  : {BASE_MODEL}")
print(f"  LoRA rank   : r={LORA_RANK}, alpha={LORA_ALPHA}  (ratio={LORA_ALPHA/LORA_RANK:.1f})")
print(f"  Epochs      : {NUM_EPOCHS}")
print(f"  Batch (eff) : {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")
print(f"  LR          : {LEARNING_RATE}")
print(f"  QA cache    : {QA_CACHE}")
print(f"  Output      : {OUTPUT_DIR.resolve()}")

# ── VRAM check ─────────────────────────────────────────────────────────────────
free_vram = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved()) / 1e9
if free_vram < 12:
    print(f"\n\u26a0\ufe0f  WARNING: Only {free_vram:.1f} GB free VRAM.  The 9B model needs ~8.5 GB")
    print("   for loading plus ~3-5 GB for training.  Consider closing other GPU processes.")
else:
    print(f"\nVRAM available: {free_vram:.1f} GB  \u2714")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch  : 2.10.0+cu128
CUDA     : True  |  GPU: NVIDIA GeForce RTX 4090
VRAM     : 25.4 GB

Configuration:
  Base model  : unsloth/Qwen3.5-9B
  LoRA rank   : r=32, alpha=32  (ratio=1.0)
  Epochs      : 3
  Batch (eff) : 2 x 4 = 8
  LR          : 0.0002
  QA cache    : qa_dataset/peterson_qa.jsonl
  Output      : /home/rob/PythonEnvironments/FineTuning/FineTuning/NoteBooks/JordanPeterson/outputs/qwen3_5_9b_peterson_lora

VRAM available: 25.4 GB  ✔


---
# Part 1: Load the Q&A Dataset

Same V4 cache used by `Qwen3_14B_JordanPeterson_V4_FineTuning.ipynb`:
**3,936 pairs** from **1,968 passages** with both front-matter AND back-matter
removed.

In [2]:
from collections import Counter

if not QA_CACHE.exists():
    raise FileNotFoundError(
        f"Q&A cache not found at {QA_CACHE}. "
        "Run JordanPeterson_DataPrep.ipynb first to generate the dataset."
    )

with open(QA_CACHE) as f:
    records = [json.loads(line) for line in f if line.strip()]

raw_dataset = Dataset.from_list([
    {"question": r["question"], "answer": r["answer"]}
    for r in records
])

print(f"Dataset loaded: {len(raw_dataset):,} Q&A pairs")
print(f"Schema: {raw_dataset.column_names}")
print()

book_dist = Counter(r["book"] for r in records)
print("Distribution by book:")
for book, count in sorted(book_dist.items(), key=lambda x: -x[1]):
    print(f"  {book:<35}  {count:4d}  ({100*count/len(records):.1f}%)")

Dataset loaded: 5,028 Q&A pairs
Schema: ['question', 'answer']

Distribution by book:
  We Who Wrestle with God              2330  (46.3%)
  Maps of Meaning                      1042  (20.7%)
  12 Rules for Life                     878  (17.5%)
  Beyond Order                          778  (15.5%)


---
# Part 2: Load the Qwen3.5-9B Base Model

## Architecture: Hybrid Linear + Full Attention

Qwen3.5-9B uses a **Gated DeltaNet** architecture that alternates between:
- **Linear attention** layers (24 of 32): O(n) complexity, efficient for long contexts
- **Full attention** layers (8 of 32): standard quadratic attention for global reasoning

This hybrid approach gives near-linear scaling for long sequences while retaining
the quality of full attention where it matters most.

## Loading via `FastVisionModel`

All Qwen3.5 models are vision-language models (VLMs) with an integrated vision
encoder.  Even for text-only fine-tuning, we load via `FastVisionModel` which
returns a `processor` object — the tokenizer is accessed as `processor.tokenizer`.
The vision encoder weights are frozen and do not participate in LoRA training.

## VRAM Budget

| Component | Est. VRAM |
|-----------|----------|
| 9B model weights (4-bit) | ~7 GB |
| Vision encoder (frozen) | ~1 GB |
| LoRA adapters (r=32) | ~0.3 GB |
| Optimizer states (8-bit) | ~0.6 GB |
| Activations + gradients (checkpointed) | ~2-3 GB |
| **Total** | **~11-12 GB** |

With `batch_size=2` and gradient checkpointing, this fits comfortably within 24 GB.

In [3]:
print(f"Loading {BASE_MODEL} ...")

model, processor = FastVisionModel.from_pretrained(
    model_name      = BASE_MODEL,
    max_seq_length  = MAX_SEQ_LEN,
    load_in_4bit    = True,
)
tokenizer = processor.tokenizer

vram_after_load = torch.cuda.memory_reserved() / 1e9
print(f"Model loaded.  VRAM reserved: {vram_after_load:.1f} GB")
print(f"Model dtype   : {next(model.parameters()).dtype}")
print(f"Tokenizer     : {type(tokenizer).__name__}")
print(f"Free VRAM     : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved()) / 1e9:.1f} GB")

Loading unsloth/Qwen3.5-9B ...
==((====))==  Unsloth 2026.3.3: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.65 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

Model loaded.  VRAM reserved: 8.4 GB
Model dtype   : torch.bfloat16
Tokenizer     : TokenizersBackend
Free VRAM     : 16.9 GB


---
# Part 3: Add LoRA Adapters (r=32)

For Qwen3.5 VLMs, Unsloth uses a declarative API instead of explicit
`target_modules`.  We set:

- `finetune_language_layers = True` — add LoRA to all text model layers
- `finetune_vision_layers = False` — skip the vision encoder (text-only task)
- `finetune_attention_modules = True` — q, k, v, o projections
- `finetune_mlp_modules = True` — gate, up, down projections

This targets the same modules as the Qwen3-14B notebooks (`q_proj`, `k_proj`,
`v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`) but applied across
both the linear attention and full attention layer types.

In [4]:
model = FastVisionModel.get_peft_model(
    model,
    r                           = LORA_RANK,
    lora_alpha                  = LORA_ALPHA,
    lora_dropout                = 0,
    finetune_vision_layers      = False,   # skip vision encoder
    finetune_language_layers    = True,     # LoRA on all text layers
    finetune_attention_modules  = True,     # q, k, v, o projections
    finetune_mlp_modules        = True,     # gate, up, down projections
    use_gradient_checkpointing  = "unsloth",
    random_state                = 42,
    bias                        = "none",
)

# ── Count trainable parameters ─────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
pct_trainable    = 100 * trainable_params / total_params

print(f"Total parameters    : {total_params:>15,}")
print(f"Trainable (LoRA)    : {trainable_params:>15,}  ({pct_trainable:.4f}% of total)")
print(f"VRAM after LoRA     : {torch.cuda.memory_reserved()/1e9:.1f} GB")
print(f"Free VRAM           : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved()) / 1e9:.1f} GB")

Unsloth: Making `model.base_model.model.model.language_model` require gradients
Total parameters    :   5,831,582,960
Trainable (LoRA)    :      86,556,672  (1.4843% of total)
VRAM after LoRA     : 8.6 GB
Free VRAM           : 16.8 GB


---
# Part 4: Format Dataset into Qwen3.5 ChatML Format

Qwen3.5 uses the same ChatML template as Qwen3:

```
<|im_start|>system
{system prompt}<|im_end|>
<|im_start|>user
{question}<|im_end|>
<|im_start|>assistant
<think>

</think>
{answer}<|im_end|>
```

The empty `<think>\n\n</think>` block appears because we set
`enable_thinking=False`.  It is masked out during training by
`train_on_responses_only` and does not contribute to the loss.

In [5]:
def format_example(batch):
    formatted_texts = []
    for question, answer in zip(batch["question"], batch["answer"]):
        conversation = [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": question},
            {"role": "assistant", "content": answer},
        ]
        text = tokenizer.apply_chat_template(
            conversation,
            tokenize              = False,
            add_generation_prompt = False,
            enable_thinking       = False,
        )
        formatted_texts.append(text)
    return {"text": formatted_texts}


dataset = raw_dataset.map(
    format_example,
    batched=True,
    batch_size=256,
    remove_columns=raw_dataset.column_names,
)

print(f"Formatted dataset: {len(dataset):,} examples")
print(f"Columns: {dataset.column_names}")
print()
print("-" * 70)
print("SAMPLE FORMATTED TRAINING EXAMPLE")
print("-" * 70)
sample = dataset[0]["text"]
print(sample[:800])
print("...")
print("-" * 70)
print(f"Full sample length: {len(sample.split())} words / {len(sample)} chars")

Map:   0%|          | 0/5028 [00:00<?, ? examples/s]

Formatted dataset: 5,028 examples
Columns: ['text']

----------------------------------------------------------------------
SAMPLE FORMATTED TRAINING EXAMPLE
----------------------------------------------------------------------
<|im_start|>system
You are an AI assistant that has been trained on the complete works of Jordan B. Peterson, a Canadian clinical psychologist, professor, and author. You speak with deep knowledge of psychology, philosophy, mythology, religion, and personal responsibility. Your responses reflect Peterson's writing style, intellectual depth, and interdisciplinary approach to understanding human nature and meaning.<|im_end|>
<|im_start|>user
How does the brain's dual-hemisphere structure relate to the mythological opposition between order and chaos?<|im_end|>
<|im_start|>assistant
<think>

</think>

FIGURES 1 The Domain and Constituent Elements of the Known 15 2 The Metamythological Cycle of the Way 17 3 Normal Life 28 4 Revolutionary Adaptation 31 5 The Ambivale

In [6]:
# Auto-detect the response boundary tokens
_sample_text = dataset[0]["text"]

if "<|im_start|>assistant\n" in _sample_text:
    instruction_part = "<|im_start|>user\n"
    response_part    = "<|im_start|>assistant\n"
    print("Detected Qwen3.5 ChatML response boundary tokens:")
else:
    raise RuntimeError(
        "Could not detect ChatML tokens in formatted dataset. "
        "Check the output of format_example() above."
    )

print(f"  instruction_part : {repr(instruction_part)}")
print(f"  response_part    : {repr(response_part)}")

Detected Qwen3.5 ChatML response boundary tokens:
  instruction_part : '<|im_start|>user\n'
  response_part    : '<|im_start|>assistant\n'


---
# Part 5: Configure and Run Training

## Comfortable VRAM Headroom

The 9B model at 4-bit loads at ~8.5 GB, leaving ~16 GB for training overhead.
We can use the same batch/accumulation settings as the Qwen3-14B notebooks:

- **`batch_size=2`** — same as Qwen3-14B
- **`grad_accum=4`** — effective batch = 2 × 4 = 8
- **`gradient_checkpointing="unsloth"`** — Unsloth's optimised checkpointing
- **`optim="adamw_8bit"`** — 8-bit optimizer states

## Expected Step Count

With 3,936 pairs (V4 cache):
```
ceil(3936 / 2) × 3 // 4 = 1968 × 3 // 4 = 5904 // 4 = 1,476
```

Same number of gradient updates as Qwen3-14B V4.

In [7]:
import os
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"]  = "1"

sft_trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = dataset,
    args             = SFTConfig(
        dataset_text_field          = "text",
        max_length                  = MAX_SEQ_LEN,
        dataset_num_proc            = 2,

        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        num_train_epochs            = NUM_EPOCHS,

        learning_rate               = LEARNING_RATE,
        warmup_steps                = WARMUP_STEPS,
        lr_scheduler_type           = "cosine",
        weight_decay                = WEIGHT_DECAY,

        optim                       = "adamw_8bit",
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),

        logging_steps               = 25,
        save_strategy               = "epoch",
        output_dir                  = str(OUTPUT_DIR),
        report_to                   = "none",

        seed                        = 42,
        packing                     = False,
    ),
)

sft_trainer = train_on_responses_only(
    sft_trainer,
    instruction_part = instruction_part,
    response_part    = response_part,
)

# ── Masking verification ─────────────────────────────────────────────────────
_sample_input = sft_trainer.train_dataset[0]
_n_total      = len(_sample_input["input_ids"])
_n_trained    = sum(1 for lbl in _sample_input["labels"] if lbl != -100)
_n_masked     = _n_total - _n_trained
print("Response masking check on first example:")
print(f"  Total tokens   : {_n_total}")
print(f"  Trained tokens : {_n_trained}  (assistant response only)")
print(f"  Masked tokens  : {_n_masked}   (system + user = {100*_n_masked/_n_total:.0f}% of input)")
print()

_total_steps = math.ceil(len(dataset) / BATCH_SIZE) * NUM_EPOCHS // GRAD_ACCUM
print(f"Estimated gradient updates: {_total_steps:,}")
print(f"  ({len(dataset):,} examples / {BATCH_SIZE} batch x {GRAD_ACCUM} accum x {NUM_EPOCHS} epochs)")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/5028 [00:00<?, ? examples/s]

Map (num_proc=20):   0%|          | 0/5028 [00:00<?, ? examples/s]

Filter (num_proc=20):   0%|          | 0/5028 [00:00<?, ? examples/s]

Response masking check on first example:
  Total tokens   : 746
  Trained tokens : 644  (assistant response only)
  Masked tokens  : 102   (system + user = 14% of input)

Estimated gradient updates: 1,885
  (5,028 examples / 2 batch x 4 accum x 3 epochs)


In [ ]:
vram_before = torch.cuda.memory_reserved() / 1e9
print(f"VRAM before training: {vram_before:.1f} GB")
print(f"Starting training \u2014 {NUM_EPOCHS} epochs, ~{_total_steps:,} gradient updates...")
print()

t0 = time.time()
train_result = sft_trainer.train()
elapsed_min  = (time.time() - t0) / 60
vram_peak    = torch.cuda.max_memory_reserved() / 1e9

print()
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"  Steps          : {train_result.global_step:,}")
print(f"  Training loss  : {train_result.training_loss:.4f}")
print(f"  Elapsed        : {elapsed_min:.1f} min")
print(f"  Peak VRAM      : {vram_peak:.1f} GB")
print()
print("Qwen3-14B V4 reference:")
print("  Steps: 1,476  |  Loss: 1.6421  |  Time: 98.0 min  |  VRAM: 13.9 GB")

# 216m 59.6s

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


VRAM before training: 8.6 GB
Starting training — 3 epochs, ~1,885 gradient updates...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,028 | Num Epochs = 3 | Total steps = 1,887
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 86,556,672 of 9,496,370,416 (0.91% trained)


Step,Training Loss
25,2.805070
50,2.568792
75,2.544002
100,2.492445
125,2.357742
150,2.421573
175,2.352498
200,2.322825
225,2.266868
250,2.278855



TRAINING COMPLETE
  Steps          : 1,887
  Training loss  : 1.1604
  Elapsed        : 217.0 min
  Peak VRAM      : 17.4 GB

Qwen3-14B V4 reference:
  Steps: 1,476  |  Loss: 1.6421  |  Time: 98.0 min  |  VRAM: 13.9 GB


---
# Part 6: Save the LoRA Adapter

We save only the LoRA adapter weights.  The 9B model has fewer layers (32 vs 40)
and a comparable hidden dimension (4096 vs 4096) to Qwen3-14B, so the adapter
size should be somewhat smaller.

**Output path**: `outputs/qwen3_5_9b_peterson_lora/`

In [9]:
print(f"Saving LoRA adapter to {OUTPUT_DIR} ...")

model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

adapter_files = (
    list(OUTPUT_DIR.glob("*.safetensors"))
    + list(OUTPUT_DIR.glob("*.bin"))
)
total_mb = sum(f.stat().st_size for f in adapter_files) / 1e6

print()
print("Adapter files:")
for f in sorted(adapter_files):
    print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")

print()
print(f"Total adapter size: {total_mb:.1f} MB")
print(f"  (Qwen3-14B V4 r=32 was 513.9 MB)")

Saving LoRA adapter to outputs/qwen3_5_9b_peterson_lora ...

Adapter files:
  adapter_model.safetensors  (346.3 MB)

Total adapter size: 346.3 MB
  (Qwen3-14B V4 r=32 was 513.9 MB)


---
# Part 7: Inference Test

Same 5 evaluation prompts used across all fine-tuning notebooks for direct
comparison.  Greedy decoding (`do_sample=False`) for deterministic output.

In [ ]:
FastVisionModel.for_inference(model)

EVAL_PROMPTS = [
    "What is the relationship between order and chaos in human experience?",
    "Why is personal responsibility the foundation of a meaningful life?",
    "How do ancient myths and stories reveal truths about human nature?",
    "What does it mean to pursue what is meaningful rather than what is expedient?",
    "How should a person confront suffering rather than flee from it?",
]


def ask(question: str, max_new_tokens: int = 300) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            do_sample          = False,
            temperature        = 1.0,
            repetition_penalty = 1.1,
        )

    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    response   = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    response   = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()
    return response


print("Testing Qwen3.5-9B fine-tuned model (greedy decoding)...")
print()
for i, prompt in enumerate(EVAL_PROMPTS):
    print("-" * 70)
    print(f"Q{i+1}: {prompt}")
    print("-" * 70)
    answer = ask(prompt)
    print(answer if answer.strip() else "(empty response)")
    print()

    # 2m 16.6s

Testing Qwen3.5-9B fine-tuned model (greedy decoding)...

----------------------------------------------------------------------
Q1: What is the relationship between order and chaos in human experience?
----------------------------------------------------------------------
of the future self). The same thing appears true of “unconscious” processing in general: much of what we do most competently can be done without imaging or formal conceptualization because the appropriate behavior has been mapped onto the unknown in the form of habit (or rut, when negative). This means that the information contained in the image or idea does not have to be translated into action to constitute genuine knowledge—rather, it has already been so translated, and embodied. This is why classical empiricists considered the map a mere representation of reality, rather than something secondary, intermediate to the real world itself. We see this reflection even at the level of the material brain: those areas res

---
# Conclusions

## What This Notebook Tests

This is the first fine-tuning experiment with a **Qwen3.5** model in this
repository.  It tests whether the newer hybrid-attention architecture produces
different stylistic adaptation on the same training data, despite being smaller
(9B vs 14B) than the Qwen3 model used in V1-V4.

## Key Comparisons

| Aspect | Qwen3-14B V4 | **Qwen3.5-9B** |
|--------|-------------|---------------|
| Parameters | 14B | 9B |
| Architecture | Dense transformer | Hybrid (DeltaNet + full attention) |
| Layers | 40 | 32 (24 linear + 8 full) |
| Training data | 3,936 Q&A pairs | Same cache |
| LoRA rank | r=32 | r=32 |
| Effective batch | 8 | 8 |
| 4-bit VRAM | ~11 GB | ~8.5 GB |

## Expected Differences

1. **Potentially higher loss** — a smaller model has less capacity to memorise
   the training corpus, which could mean higher loss but also less overfitting

2. **Faster training** — fewer layers and narrower dimensions mean each step
   should be faster than Qwen3-14B

3. **Newer architecture benefits** — Qwen3.5's hybrid linear+full attention and
   improved pre-training may compensate for the smaller parameter count

## Adding to Comparison Notebooks

To include Qwen3.5-9B in the AllModels comparison notebook:

1. Add `"qwen3_5_9b"` to `MODEL_KEYS`
2. Add the model path: `"qwen3_5_9b": "./outputs/qwen3_5_9b_peterson_lora"`
3. Add display name, colour, and system prompt entries
4. Add a new inference phase using `FastVisionModel` loading
5. Delete the relevant pkl file and re-run

Note: the comparison notebook will need a Qwen3.5-specific inference wrapper
since loading uses `FastVisionModel` instead of `FastLanguageModel`.

In [11]:
!nvidia-smi

Fri Mar  6 11:40:36 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.288.01             Driver Version: 535.288.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 4090        Off | 00000000:04:00.0 Off |                  Off |
|  0%   39C    P8              11W / 450W |   9292MiB / 24564MiB |     11%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--